# Introduccion a brightway - pt. 2

En esta seccion hablaremos de los conceptos fundamentales de brigthway. Es importante aclarar que toda esta informacion esta disponible en linea en la pagina de documentacion: 

https://docs.brightway.dev/en/latest/index.html

## Exportar bases de datos y proyectos
En la seccion anterior aprendimos a crear bases de datos de manera automatica ('biosphere3') y de manera manual ('mi_base_de_datos'). 
En situaciones convencionales, es normal que necesitemos compartir nuestros modelos de inventario, ya sea durante el trabajo colaborativo o para reportar nuestro trabajo a revisores, colegas y cualquier por razones de transparencia.
Para esto, bw2io ofrece una serie de herramientas que pueden usarse para exportar los modelos en diferentes formatos. 
Por un tema de popularidad, en esta seccion nos enfocaremos en 3 herramientas:
- Exportar una base de datos a excel
- Exportar una base de datos a csv (dataframe)
- Exportar un proyecto como archivo comprimido de respaldo.

###  Exportar a excel
Brightway utiliza un template para leer y exportar bases de datos en formato excel. Es conveniente para distribuir versiones finales del inventario. No es muy bueno almacenando informacion anidad. No permite 'trackear' los cambios debido a que *.xlsx no es un formato de texto.

In [1]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print
# Primero que nada, verifiquen que esten en el proyecto adecuado
bd.projects

Brightway2 projects manager with 5 objects:
	default
	nuevo_nuevo_proyecto_2
	nuevo_proyecto_2
	peru25
	peru25-prueba
Use `projects.report()` to get a report on all projects.

In [3]:
# Si no es el proyecto adecuado, ya saben que hacer
bd.projects.set_current('peru25')

In [5]:
bd.databases

Databases dictionary with 2 object(s):
	biosphere3
	mi_base_de_datos

In [6]:
# dirpath es el argumento que controla en que ubicacion se exportara el archivo. 
# En sistemas operativos tipo UNIX (Linux, MacOS), '.' significa 'aqui'.
directorio = bi.export.excel.write_lci_excel(database_name='mi_base_de_datos',dirpath='.')

In [7]:
directorio

'./lci-mi_base_de_datos.xlsx'

###  Exportar a csv
Brightway permite convertir los nodos (actividades) y aristas (exchanges) en DataFrames de [pandas](https://pandas.pydata.org/).
Un DataFrame es un estructura de datos tabular que es muy usada en analisis y ciencia de datos, y puede ser exportada directamente como archivo CSV.


In [9]:
db = bd.Database('mi_base_de_datos')

In [10]:
db.nodes_to_dataframe() # Solo los nodos

,code,database,id,location,name,unit
0,ng,mi_base_de_datos,4712,NO,Nat Gas,MJ
2,CF,mi_base_de_datos,4711,CN,carbon fibre,kilogram
1,bici,mi_base_de_datos,4710,PE,produccion bici,piece


In [11]:
db.edges_to_dataframe() # Solo aristas

Getting activity data


100%|██████████| 3/3 [00:00<00:00, 34192.70it/s]


Adding exchange data to activities


100%|██████████| 4/4 [00:00<00:00, 4607.86it/s]


Filling out exchange data


100%|██████████| 3/3 [00:00<00:00, 1863.03it/s]

Creating DataFrame
Compressing DataFrame


,target_id,target_database,target_code,target_name,target_reference_product,target_location,target_unit,target_type,source_id,source_database,source_code,source_name,source_product,source_location,source_unit,source_categories,edge_amount,edge_type
0,4711,mi_base_de_datos,CF,carbon fibre,NaN,CN,kilogram,process,4712,mi_base_de_datos,ng,Nat Gas,NaN,NO,MJ,NaN,237.300000,technosphere
1,4711,mi_base_de_datos,CF,carbon fibre,NaN,CN,kilogram,process,4713,biosphere3,co2,Carbon Dioxide,NaN,GLO,NaN,air,0.112236,biosphere
2,4711,mi_base_de_datos,CF,carbon fibre,NaN,CN,kilogram,process,4714,biosphere3,n2o,monoxido dinitrogeno,NaN,GLO,NaN,air,0.230000,biosphere
3,4710,mi_base_de_datos,bici,produccion bici,NaN,PE,piece,process,4711,mi_base_de_datos,CF,carbon fibre,NaN,CN,kilogram,NaN,2.500000,technosphere


In [12]:
# La funcion `to_csv` es propia de pandas, no de brightway
db.nodes_to_dataframe().to_csv('mis-nodos.csv')
db.edges_to_dataframe().to_csv('mis-aristas.csv')

Getting activity data


100%|██████████| 3/3 [00:00<00:00, 40459.52it/s]


Adding exchange data to activities


100%|██████████| 4/4 [00:00<00:00, 4855.92it/s]


Filling out exchange data


100%|██████████| 3/3 [00:00<00:00, 1996.65it/s]

Creating DataFrame
Compressing DataFrame


###  Exportar proyecto completo como backup
La ultima opcion mas comun es la de exportar el proyecto completo en forma de archivo comprimido. Esto suele hacer cuando se desea guardar copias de todas las bases de datos de un proyecto. La desventaja es que el archivo resultado puede ser pesado y no es adecuado si no se tienen los permisos para compartir bases de datos comerciales.

In [13]:
bi.backup_project_directory('peru25',dir_backup='.')

Creating project backup archive - this could take a few minutes...
Saved to: brightway2-project-peru25-backup03-February-2025-10-29PM.tar.gz


PosixPath('brightway2-project-peru25-backup03-February-2025-10-29PM.tar.gz')

## Importar bases de datos privadas
Esta seccion es una continuacion natural de la anterior ya que simplemente aprenderemos a importar los archivos que fueron exportados previamente. Asumiremos, nuevamente, que excel, csv, y backup.tar.gz son los unicos formatos que nos interesan.

### Importar un archivo de excel

In [14]:
importador = bi.ExcelImporter('lci-mi_base_de_datos.xlsx')
importador.apply_strategies()
importador.match_database(fields=('name', 'code', 'unit', 'location'))  # Conecta nodos del archivo excel
importador.match_database('biosphere3', fields=('name','unit','categories')) # Conecta nodos con la base de datos biosphere3
importador.statistics()
importador.write_excel()

Extracted 1 worksheets in 0.01 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 8.75 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
3 datasets
4 exchanges
0 unlinked exchanges
  
Wrote matching file to:
/home/jupyter-su

'/home/jupyter-summer25peru_glarr-180f6/.local/share/Brightway3/peru25.01fb3473/output/db-matching-mi_base_de_datos.xlsx'

In [15]:
bd.databases

Databases dictionary with 2 object(s):
	biosphere3
	mi_base_de_datos

### Repliquemos los resultados
Ahora podemos 'simular' un ejercicio de reproducibilidad, y realizar el calculo de los impactos una vez mas.

In [16]:
db = bd.Database('mi_base_de_datos')
bicicleta = db.get('bici') # seleccionamos la actividad que tiene codigo 'bici', la definimos en la seccion anterior

In [17]:
lca = bc.LCA({bicicleta:1},method=('dummy',)) # Instancia la clase
lca.lci() # calcula el inventario de ciclo de vida
lca.lcia() # Calcula los impactos 
print("El impacto es: ", lca.score) # Es el mismo 🎉

El impacto es:  159.21749937877053

### Importar el backup del proyecto
Este modalidad no require mucha explicacion: El proyecto se carga nuevamente. 

In [19]:
bi.restore_project_directory(
    'brightway2-project-peru25-backup03-February-2025-10-29PM.tar.gz',  # nombre del archivo, creado celdas arriba
    project_name='nuevo_nuevo_proyecto_2', # Se puede elegir un nombre nuevo para el proyecto
    overwrite_existing = False
    )

Restoring project backup archive - this could take a few minutes...
Restored project: nuevo_nuevo_proyecto_2


'nuevo_nuevo_proyecto_2'

In [20]:
bd.projects

Brightway2 projects manager with 5 objects:
	default
	nuevo_nuevo_proyecto_2
	nuevo_proyecto_2
	peru25
	peru25-prueba
Use `projects.report()` to get a report on all projects.

🚧 **Manos a la obra**:
- Un colega ha encontrado un error en tu modelo. La cantidad de Gas Natural consumida por la fibra de carbono no es 237.3, sino 23.73
- Descarga el archivo de excel `lci-mi_base_de_datos.xlsx` a tu computadora personal y modifica el valor manualmente.
- Importa el archivo excel modificado y vuelve a calcular el ACV. Cuanto ha cambiado el impacto final?

In [ ]:
# Tu codigo aqui

## Importar bases de datos comerciales
Hemos aprendido a construir un modelo de ACV desde cero y de forma manual. Aunque esto resulta bastante util, en la realidad solemos combinar nuestros datos con aquellos provenientes de bases de datos comerciales. En esta seccion nos enfocaremos en la base de datos ecoinvent (v3.9), que es una de las mas utilizadas en el sector. 

En la actualidad hay dos maneras de importar los datos de ecoinvent en nuestro proyecto:
- Leyendo los archivos ecospold2 crudos directamente del disco y convirtiendolos en una base de datos de brigthway.
- Utilizando la herramienta `import_ecoinvent_release` que descarga la base de datos desde un servidor remoto.
  
### Importando ecoinvent (crudo) desde el disco

Para este caso, es necesario haber descargado ecoinvent. Ecoinvent es distribuido en formato comprimido 7z, y contiene todas las actividades en formato ecospold2 (algo similar a XML). `bw2io` tiene funciones disenadas para interpretar la informacion, verificar que los `exchanges` sean correctos, y que los nodos de la biosfera existan en la base de datos 'biosphere3'.


In [21]:
# Los archivos ecospold2 se ven asi:
!ls /media/ei391/datasets | head

0002742c-69b1-5fa0-ab0a-314f62b58487_66c93e71-f32b-4591-901c-55395db5c132.spold
0004576b-700d-5d1a-8114-c3b71b964493_759b89bd-3aa6-42ad-b767-5bb9ef5d331d.spold
00095440-d74e-5bc7-bdab-424e2aa0ab62_3fdbecc2-bb86-4c4b-8f5c-7b80e5e365c8.spold
0010d015-9a82-5171-b08b-9bd4329bfc38_66c93e71-f32b-4591-901c-55395db5c132.spold
00127df5-eea7-5773-857f-589689ba61fd_66c93e71-f32b-4591-901c-55395db5c132.spold
001543cb-1692-51f5-ad6a-d06d04806b50_27da8130-82ba-485c-a800-b89efdcb0491.spold
0015d952-92b4-5295-b22a-0adf349f595d_1be4f7e4-5244-4f9d-b80d-7fbf1e337e2b.spold
00188ca8-d204-59af-983b-713e2d97d1d8_58b17444-9524-4a28-8ff5-58b7d6328dd2.spold
00203816-11e5-58ab-b58a-549eead26b37_7c4dafff-fe18-45c0-92e4-857950032abb.spold
0020a7a8-9e7a-5025-8842-99eba36b590f_66c93e71-f32b-4591-901c-55395db5c132.spold
ls: write error: Broken pipe


In [22]:
# Para importar, hay que seguir los siguentes pasos:
# 1. Leer los archivos XML e dejar que brigthway los interprete.
db = bi.SingleOutputEcospold2Importer(dirpath='/media/ei391/datasets',db_name='ecoinvent39')

Extracting XML data from 21238 datasets
Extracted 21238 datasets in 1104.16 seconds


In [23]:
# 2. Aplicar una serie de estrategias para asegurarse que no existe informacion corrupta y que la importacion es posible
db.apply_strategies()

Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: remove_zero_amount_coproducts
Applying strategy: remove_zero_amount_inputs_with_no_activity
Applying strategy: remove_unnamed_parameters
Applying strategy: es2_assign_only_product_with_amount_as_reference_product
Applying strategy: assign_single_product_as_activity
Applying strategy: create_composite_code
Applying strategy: drop_unspecified_subcategories
Applying strategy: fix_ecoinvent_flows_pre35
Applying strategy: drop_temporary_outdated_biosphere_flows
Applying strategy: link_biosphere_by_flow_uuid
Applying strategy: link_internal_technosphere_by_composite_code
Applying strategy: delete_exchanges_missing_activity
Applying strategy: delete_ghost_exchanges
Applying strategy: remove_uncertainty_from_negative_loss_exchanges
Applying strategy: fix_unreasonably_high_lognormal_uncertainties
Applying strategy: convert_activity_parameters_to_list
Applying strategy: add_cpc_classification_from

In [ ]:
# 3. Ecoinvent esta listo en la memoria pero aun no ha sido grabado en el disco. 
# Hay que grabarlo en el disco.
db.write_database()

Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 21238/21238 [00:33<00:00, 632.85it/s]


Vacuuming database 


Para verificar que ha sido importado correctamente, podemos repetir el ejercicio realizado con la base de datos 'biosphere3' de la anterior seccion.bd.databases

In [ ]:
bd.databases # Lista de las bases de datos

In [ ]:
ei = bd.Database('ecoinvent39')
len(ei) # Muestra la cantidad de elementos

### Importando ecoinvent desde un servidor remoto
Para este caso utilizamos la funcion `bw2io.import_ecoinvent_release` que se encarga de 1) instalar una biosfera, 2) instalar los metodos de impacto mas actuales, y 3) instalar la base de datos ecoinvent. 
Como podran imaginar, requiere la autenticacion del usuario que debe poseer un cuenta de acceso ecoinvent

In [ ]:
# bw2io.import_ecoinvent_release(
#     version="3.9" 
#     system_model="cutoff", # Otras opciones son: "consequential", "apos" y "EN15804"
#     username="xxxx", # Tu usuario
#     password="xxxx", # Tu clave
#     biosphere_name="biosphere3" # Optional, puedes guardar la base de datos de la biosfera con otro nombre.
# )


In [2]:
bi.restore_project_directory('/media/ei391/brightway2-project-peru25-ei-3.9.1-cutoff-backup.tar.gz') # Este archivo contiene el proyecto
bd.projects.set_current('peru25-ei-3.9.1-cutoff')

Restoring project backup archive - this could take a few minutes...
Restored project: peru25-ei-3.9.1-cutoff


In [3]:
bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.9.1-biosphere
	ecoinvent-3.9.1-cutoff

In [4]:
ei = bd.Database("ecoinvent-3.9.1-cutoff")

In [19]:
seleccionado = ei.random() # Explora las actividades
seleccionado

'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)

In [15]:
print(seleccionado.as_dict())

{
    'comment': "This inventory describes the supply of nutrients from ''potassium nitrate'' for fertiliser use.\nNo
machinery expenditures or emissions to the environment are considered related to the actual application of this 
substance on soil during its agricultural use. The purpose of this activity is to connect the chemical production 
of ''potassium nitrate'' with its generic uses as fertilizer. The product ''potassium nitrate'' is connected to the
generic market for ''inorganic nitrogen fertiliser, as N'' and ''inorganic potassium fertiliser, as K2O'' through 
this supplying activity. In the agricultural context, ''potassium nitrate'' can be used directly as a fertilizer 
input to crop production, eventually with other chemicals; alternatively, a generic input of ''inorganic nitrogen 
fertiliser, as N'' and ''inorganic potassium fertiliser, as K2O'' can be used instead, and this will represent the 
regional consumption mix for his type of nutrient. A way in between is allowed by using the generic input 
''inorganic nitrogen fertiliser, as N'' and ''inorganic potassium fertiliser, as K2O'' and link to the activity 
providing it from a specific chemical.\nIn all cases, emissions associated to the application of the fertilizer 
have to be calculated in the crop production activity.\nImage: 
https://db3.ecoinvent.org/images/e8342b95-78fd-4a83-a444-47e7464508b4\nIncluded activities start:  This inventory 
describes the supply of nutrients from ''potassium nitrate'' for fertiliser use.\nIncluded activities end:  No 
machinery expenditures or emissions to the environment are considered related to the actual application of this 
substance on soil during its agricultural use.\nTechnology:  Inventory based on composition of ''potassium 
nitrate''.",
    'classifications': [
        ('ISIC rev.4 ecoinvent', '2012:Manufacture of fertilizers and nitrogen compounds'),
        ('CPC', '3463: Mineral or chemical fertilizers, potassic')
    ],
    'activity type': 'ordinary transforming activity',
    'activity': '5661d278-5f53-524a-b37d-f81a0facfbc5',
    'database': 'ecoinvent-3.9.1-cutoff',
    'filename': '5661d278-5f53-524a-b37d-f81a0facfbc5_7c4dafff-fe18-45c0-92e4-857950032abb.spold',
    'location': 'RNA',
    'name': 'nutrient supply from potassium nitrate',
    'synonyms': [],
    'parameters': [],
    'authors': {
        'data entry': {'name': 'Avraam Symeonidis', 'email': 'Symeonidis@ecoinvent.org'},
        'data generator': {'name': 'Avraam Symeonidis', 'email': 'Symeonidis@ecoinvent.org'}
    },
    'type': 'process',
    'reference product': 'inorganic potassium fertiliser, as K2O',
    'flow': '7c4dafff-fe18-45c0-92e4-857950032abb',
    'unit': 'kilogram',
    'production amount': 1.0,
    'code': '39b3113bf30886d67d8044ddecea5748',
    'id': 23385
}

In [9]:
my_act = ei.get("25d47ea413b0aa231026b5c31e293962")
my_act

'market for sodium sulfate, anhydrite' (kilogram, RoW, None)

Como pueden notar, el contenido de la actividad ecoinvent es bastante rica. Existen campos fuera de `name`, `code`,`location` y `unit` que son nuevos para nosotros, lo que demuestra que brightway es lo suficientemente flexible al definir una actividad. 

Lo que vimos en la celda anterior describe a una actividad, pero aun no describe sus conexiones (`exchanges`). Para acceder a ellas, hay que utilizar las funciones `exchanges`, `technosphere` o `biosphere`, segun lo que se desee observar.

In [20]:
list(seleccionado.exchanges())

[Exchange: 1.0 kilogram 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
 Exchange: 0.0076 kilogram 'market for acrylonitrile-butadiene-styrene copolymer' (kilogram, GLO, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
 Exchange: 0.0001 kilogram 'market for aluminium, wrought alloy' (kilogram, GLO, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
 Exchange: 0.0012 kilogram 'market for barite' (kilogram, GLO, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
 Exchange: 0.00020082229450820924 kilogram 'market for calcium carbide, technical grade' (kilogram, RoW, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
 Exchange: 9.917770549179075e-05 kilogram 'market for calcium carbide, technical grade' (kilogram, R

In [ ]:
# `exchanges` retorna un objeto the brightway que no es nativo de python
type(seleccionado.exchanges())

In [ ]:
# Si deseamos leerlo al estilo de una lista, hay que convertirlo en una lista.
print(list(seleccionado.exchanges()))

In [21]:
# Si deseamos solo la tecnosfera, usamos la funcion correspondiente
print(list(seleccionado.technosphere()))

[
    Exchange: 0.0076 kilogram 'market for acrylonitrile-butadiene-styrene copolymer' (kilogram, GLO, None) to 
'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.0001 kilogram 'market for aluminium, wrought alloy' (kilogram, GLO, None) to 'battery production, 
lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.0012 kilogram 'market for barite' (kilogram, GLO, None) to 'battery production, lead acid, 
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.00020082229450820924 kilogram 'market for calcium carbide, technical grade' (kilogram, RoW, None) 
to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 9.917770549179075e-05 kilogram 'market for calcium carbide, technical grade' (kilogram, RER, None) to
'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.0004 kilogram 'market for carbon black' (kilogram, GLO, None) to 'battery production, lead acid, 
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.005 kilogram 'market for copper, cathode' (kilogram, GLO, None) to 'battery production, lead acid, 
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.013708994484572521 kilowatt hour 'market for electricity, medium voltage' (kilowatt hour, AU, None)
to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.20064942020073778 kilowatt hour 'market group for electricity, medium voltage' (kilowatt hour, RER,
None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.8277814450904746 kilowatt hour 'market group for electricity, medium voltage' (kilowatt hour, RAS, 
None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.09022060983665736 kilowatt hour 'market group for electricity, medium voltage' (kilowatt hour, RLA,
None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.061362019059482506 kilowatt hour 'market for electricity, medium voltage' (kilowatt hour, RU, None)
to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.033312284988490974 kilowatt hour 'market group for electricity, medium voltage' (kilowatt hour, CA,
None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.045278458758049325 kilowatt hour 'market group for electricity, medium voltage' (kilowatt hour, 
RAF, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.002686767581533908 kilowatt hour 'market for electricity, medium voltage' (kilowatt hour, NZ, None)
to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.025 kilogram 'market for glass fibre' (kilogram, GLO, None) to 'battery production, lead acid, 
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 6.0 megajoule 'market group for heat, district or industrial, natural gas' (megajoule, GLO, None) to 
'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.6200000000000001 megajoule 'market group for heat, district or industrial, other than natural gas' 
(megajoule, GLO, None) to 'battery production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 2e-06 kilogram 'market for integrated circuit, logic type' (kilogram, GLO, None) to 'battery 
production, lead acid, rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.71 kilogram 'market for lead' (kilogram, GLO, None) to 'battery production, lead acid, 
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 4.58e-10 unit 'market for metal working factory' (unit, GLO, None) to 'battery production, lead acid,
rechargeable, stationary' (kilogram, RoW, None)>,
    Exchange: 0.07

La impresion realizada en la celda de arriba nos muestra la informacion necesaria para poder construir las matrices. Sin embargo, brightway nos permite manipular el `exchange` y acceder a su metadata.

In [22]:
# Seleccionamos el segundo `exchange`de la lista
exchange = list(seleccionado.technosphere())[1]
print(exchange.as_dict())

{
    'flow': 'fef44ccb-917e-4c8d-bb35-a1898827b659',
    'type': 'technosphere',
    'name': 'aluminium, wrought alloy',
    'classifications': {'CPC': ['4153: Semi-finished products of aluminium or aluminium alloys']},
    'production volume': 0.0,
    'properties': {
        'carbon allocation': {'amount': 0.0, 'unit': 'kg'},
        'carbon content': {'amount': 0.0, 'unit': 'dimensionless'},
        'carbon content, fossil': {'amount': 0.0, 'comment': 'Aluminium', 'unit': 'dimensionless'},
        'carbon content, non-fossil': {'amount': 0.0, 'comment': 'Aluminium', 'unit': 'dimensionless'},
        'dry mass': {'amount': 1.0, 'unit': 'kg'},
        'water content': {'amount': 0.0, 'comment': 'water mass/dry mass', 'unit': 'dimensionless'},
        'water in wet mass': {'amount': 0.0, 'unit': 'kg'},
        'wet mass': {'amount': 1.0, 'unit': 'kg'}
    },
    'activity': '7ac71fd7-a65b-5f2c-9289-7335f9945c11',
    'unit': 'kilogram',
    'comment': 'Grid alloying additive; original material: Aluminium, production mix, at plant/RER Source: Spanos 
et al. (2015, Tables 5 and 6)',
    'amount': 0.0001,
    'pedigree': {
        'reliability': 3,
        'completeness': 4,
        'temporal correlation': 4,
        'geographical correlation': 5,
        'further technological correlation': 1
    },
    'uncertainty type': 2,
    'loc': -9.210340371976182,
    'scale': 0.12083045973594572,
    'scale without pedigree': 0.02449489742783178,
    'input': ('ecoinvent-3.9.1-cutoff', 'd25c8e0755ee9899fb4e892990397a68'),
    'output': ('ecoinvent-3.9.1-cutoff', 'ec0ae74eede2f9b127bbc8c4087a819c')
}

## Opciones de busqueda
Como podran imaginar, manipular una base de datos con tantas actividades (~21k) es bastante complicado. Podemos utilizar funciones nativas de python (list comprehension) para realizar una busqueda.

In [23]:
truck = [x for x in ei if x['name'] == 'transport, freight, lorry >32 metric ton, EURO5'][0]
truck

'transport, freight, lorry >32 metric ton, EURO5' (ton kilometer, BR, None)

In [24]:
print(truck.as_dict())

{
    'comment': "This dataset is an adaptation of “transport, freight, lorry >32 metric ton, EURO5” in Europe, as 
available in version 3.6 of the ecoinvent database to reflect the situation in Brazil. It represents the service of
1tkm freight transport in a lorry of the size class >32 metric tons gross vehicle weight (GVW) and Euro 5 emissions
class.The Brazilian lorry fleet is regulated under the Vehicles Air Pollution Control Program – Proconve, which 
phases are equivalent to the European control program – EURO. Since 2012, the Proconve P7 (EURO 5) phase is in 
practice, while the P8 phase (EURO 6) will start in 2023. Before that, the Proconve P6 phase (EURO 4) was not 
implemented because of the unavailability of low-sulphur diesel, therefore recontextualized datasets do not 
consider this phase. The P5 (EURO 3), P4 (EURO 2) and P3 (EURO 1) phases started in 2005, 2000 and 1996, 
respectively. Prior technologies are classified as unregulated.\nFor the dataset recontextualization to the 
Brazilian reality, an updated average freight load and the diesel with 10 ppm of sulfur and 12% biodiesel blend are
considered. Moreover, data from emission tests of the national vehicle production and import (CETESB, 2019) is used
to update regulated emissions (carbon monoxide, particulate matter and nitrogen oxides). Furthermore, correction 
factors are used to consider the impact of biodiesel blend on exhaust emissions (USEPA 2002), and the fuel 
composition is considered to account for carbon dioxide and sulphur dioxide emission.\nThe vehicle mass category 
classification considered in Brazilian national statistics is approximated to the one adopted in ecoinvent 
datasets. The larger than 32 metric ton lorry is representing the Brazilian heavy-duty lorry with gross vehicle 
weight (GVW)  larger than 15 metric tons and combined gross vehicle weight (CGVW) larger than 40-ton category 
classification. For the Brazilian classification, CGVW refers to the total weight of the combinations of vehicles, 
i.e. trailers. The average capacity utilization factor (including empty trips) for this category is 65.2 % 
according to the Road Freight Transport Model from the Brazilian Energy Research Enterprise – EPE (Stukart, 2018). 
Whereas, the average payload capacity for this category is 27.2 ton (Novo, 2016), resulting in an average freight 
load of 17.7 ton. GWV is estimated by assuming the same empty vehicle weight as for the RER region for the 
respective matching categories and accounting for the updated freight load. This resulted in a GWV of 35 ton. 
Vehicle mass dependent non-exhaust emissions (i.e. tyre, brake and road wear) are adjusted accordingly.\nThe 
emissions of carbon monoxide (CO), nitrogen oxides (NOx) and Particulate Matter (PM) were updated with data from 
(CETESB, 2019), which uses data from emission testing of the national vehicle fleet production and imports, 
weighted by sales amounts. Those tests are run with a reference fuel, which is not blended with biodiesel (ANP, 
2018), therefore, those emission factors are adjusted for emissions from burning biodiesel.\nThe impact of the 12% 
biodiesel blend in exhaust emissions is accounted for by correction factors derived from  USEPA (2002). Correction 
factors were calculated for the emissions of nitrogen oxides, particulate matter, hydrocarbons, carbon monoxide, 
acetaldehyde, ethylbenzene, formaldehyde, naphthalene and xylene. Moreover, fuel consumption was corrected with 
energy content values. For conventional diesel, it was considered energy content of 129.500 Btu/gal, animal-based 
biodiesel 115.720 Btu/gal and plant-based biodiesel 119.216 Btu/gal (USEPA 2002).\nFuel dependent emissions were 
updated as well. EURO V lorries are fuelled with 10 ppm sulfur content diesel (blended with 12% biodiesel). 
Therefore, sulphur dioxide emissions were corrected assuming that all sulphur is emitted as SO2 (0.00002 kg SO2/kg 
fossil diesel) and to account for the blend of biodiesel, w

Esta manera de buscar es mas 'pythonic'. Sin embargo, tambien puedes usar el buscador de brightway a traves de la funcion `search`.

In [31]:
ei.search('transport, freight RoW lorry 16-32 EURO5')[2]

'transport, freight, lorry >32 metric ton, EURO5' (ton kilometer, RoW, None)

In [26]:
# La funcion search prioriza algunos campos para hacer el filtro.
ei.search??


Signature: ei.search(string, **kwargs)
Source:   
    def search(self, string, **kwargs):
        """Search this database for ``string``.

        The searcher include the following fields:

        * name
        * comment
        * categories
        * location
        * reference product

        ``string`` can include wild cards, e.g. ``"trans*"``.

        By default, the ``name`` field is given the most weight. The full weighting set is called the ``boost`` dictionary, and the default weights are::

            {
                "name": 5,
                "comment": 1,
                "product": 3,
                "categories": 2,
                "location": 3
            }

        Optional keyword arguments:

        * ``limit``: Number of results to return.
        * ``boosts``: Dictionary of field names and numeric boosts - see default boost values above. New values must be in the same format, but with different weights.
        * ``filter``: Dictionary of criteria that searc